In [1]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Memememe@07",
    database="cart2insights"
)

print("Connected to MySQL successfully!")


# Load required tables
orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)
order_reviews = pd.read_sql("SELECT * FROM order_reviews", conn)
order_payments = pd.read_sql("SELECT * FROM order_payments", conn)
products = pd.read_sql("SELECT * FROM products", conn)

print("Data loaded successfully!")

Connected to MySQL successfully!


C:\Users\Siddarth.b\AppData\Local\Temp\ipykernel_5428\828446316.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql("SELECT * FROM orders", conn)
C:\Users\Siddarth.b\AppData\Local\Temp\ipykernel_5428\828446316.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql("SELECT * FROM order_items", conn)
C:\Users\Siddarth.b\AppData\Local\Temp\ipykernel_5428\828446316.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_reviews = pd.read_sql("SELECT * FROM order_reviews", conn)
C

Data loaded successfully!


C:\Users\Siddarth.b\AppData\Local\Temp\ipykernel_5428\828446316.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products", conn)


In [2]:
import pandas as pd
from scipy.stats import ttest_ind

# Convert date columns
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

# Calculate delivery delay
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

# Create order-level review score
review_by_order = (
    order_reviews.groupby("order_id")["review_score"]
    .mean()
    .reset_index()
)

# Combine delivery delay and review score
test_data = orders[
    ["order_id", "delivery_delay_days"]
].merge(
    review_by_order,
    on="order_id",
    how="inner"
)

# Remove missing values
test_data = test_data.dropna(
    subset=["delivery_delay_days", "review_score"]
)

# Create two groups
on_time_reviews = test_data[
    test_data["delivery_delay_days"] <= 0
]["review_score"]

delayed_reviews = test_data[
    test_data["delivery_delay_days"] > 0
]["review_score"]

# Perform independent t-test
t_statistic, p_value = ttest_ind(
    on_time_reviews,
    delayed_reviews,
    equal_var=False
)

print("On-time/Early Orders - Average Review Score:",
      on_time_reviews.mean())

print("Delayed Orders - Average Review Score:",
      delayed_reviews.mean())

print("T-Statistic:", t_statistic)
print("P-Value:", p_value)

# Interpretation
if p_value < 0.05:
    print("Result: There is a statistically significant difference in review scores.")
else:
    print("Result: There is no statistically significant difference in review scores.")

On-time/Early Orders - Average Review Score: 4.292441279219977
Delayed Orders - Average Review Score: 2.2716243894753427
T-Statistic: 100.61023231197377
P-Value: 0.0
Result: There is a statistically significant difference in review scores.


In [4]:
# Create Total Order Value

order_items["item_total"] = (
    order_items["price"] + order_items["freight_value"]
)

order_value = (
    order_items.groupby("order_id")["item_total"]
    .sum()
    .reset_index()
)

order_value.rename(
    columns={"item_total": "total_order_value"},
    inplace=True
)

print(order_value.head())

                           order_id  total_order_value
0  00010242fe8c5a6d1ba2dd792cb16214              72.19
1  00018f77f2f0320c557190d7a144bdd3             259.83
2  000229ec398224ef6ca0657da4fc703e             216.87
3  00024acbcdf0a6daa1e931b038114c75              25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9             218.04


In [5]:
from scipy.stats import f_oneway

# Combine order items with product categories and order value
anova_data = (
    order_items[
        ["order_id", "product_id"]
    ]
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        order_value[
            ["order_id", "total_order_value"]
        ],
        on="order_id",
        how="inner"
    )
)

# Remove missing categories
anova_data = anova_data.dropna(
    subset=["product_category_name", "total_order_value"]
)

# Select top 10 categories by number of records
top_categories = (
    anova_data["product_category_name"]
    .value_counts()
    .head(10)
    .index
)

anova_data = anova_data[
    anova_data["product_category_name"].isin(top_categories)
]

# Create groups for ANOVA
groups = [
    group["total_order_value"].values
    for _, group in anova_data.groupby("product_category_name")
]

# Perform One-Way ANOVA
f_statistic, p_value = f_oneway(*groups)

print("F-Statistic:", f_statistic)
print("P-Value:", p_value)

# Interpretation
if p_value < 0.05:
    print("Result: There is a statistically significant difference in order value across product categories.")
else:
    print("Result: There is no statistically significant difference in order value across product categories.")

F-Statistic: 148.449375896473
P-Value: 2.230822601092363e-279
Result: There is a statistically significant difference in order value across product categories.


In [6]:
from scipy.stats import chi2_contingency

# Combine payment type with order status
chi_data = order_payments[
    ["order_id", "payment_type"]
].merge(
    orders[
        ["order_id", "order_status"]
    ],
    on="order_id",
    how="inner"
)

# Create contingency table
contingency_table = pd.crosstab(
    chi_data["payment_type"],
    chi_data["order_status"]
)

print("Contingency Table:")
print(contingency_table)


# Perform Chi-Square Test
chi2_statistic, p_value, degrees_of_freedom, expected = chi2_contingency(
    contingency_table
)

print("\nChi-Square Statistic:", chi2_statistic)
print("P-Value:", p_value)
print("Degrees of Freedom:", degrees_of_freedom)


# Interpretation
if p_value < 0.05:
    print(
        "Result: There is a statistically significant association "
        "between payment type and order status."
    )
else:
    print(
        "Result: There is no statistically significant association "
        "between payment type and order status."
    )

Contingency Table:
order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74586       239         224   
debit_card           0         7        0       1486         6           2   
not_defined          0         3        0          0         0           0   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
not_defined         0            0  
voucher            84           47  

Chi-Square Statistic: 677.0831369637666
P-Value: 1.2047397488650834e-124
Degrees of Freedom: 28
Result: There is a statistically significant association between payment type 